# Baseline Model A

Train the non-personalized JDsearch LightGBM ranker from the existing seven baseline features. Run cells from top to bottom in Colab. The notebook loads all four training chunks and the validation split; it neither reads nor scores the test split.

Features and labels are joined by (user_id, wid), checked, and shuffled together within each query. Old cand_pos columns are ignored. Existing feature files must have been computed correctly; joining by keys cannot detect a wrong feature formula.

Outputs go to Data/Models/Baseline: trained model, validation metrics, keyed predictions, feature importance, data fingerprints and a reusable evaluation script for Model B. Final test evaluation is performed in ModelComparison.ipynb.


In [ ]:
# Install only missing dependencies in the current notebook environment.
import importlib.util
import subprocess
import sys
missing = [name for name in ['lightgbm', 'pyarrow', 'scikit-learn']
           if importlib.util.find_spec({'scikit-learn': 'sklearn'}.get(name, name)) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

import os, json, shutil, hashlib, platform
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
from IPython.display import display

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/bt4222_project')
    LOCAL = Path('/content/jd_local/baseline_model')
except ImportError:
    BASE = Path(os.environ['JD_BASE'])
    LOCAL = Path(os.environ['JD_LOCAL']) / 'baseline_model'

FEATURE_DIR = BASE / 'Data/Features'
SAVE_DIR = BASE / 'Data/Models/Baseline'
LOCAL.mkdir(parents=True, exist_ok=True)
SEED, K = 42, 10
FEATURES = [
    'query_coverage', 'tfidf_cosine', 'brand_name_match',
    'prod_pop_ord', 'prod_pop_cart', 'prod_pop_click', 'has_meta',
]
KEYS = ['user_id', 'wid']
print('Input:', FEATURE_DIR)
print('Output:', SAVE_DIR)


Mounted at /content/drive
Input: /content/drive/MyDrive/bt4222_project/Data/Features
Output: /content/drive/MyDrive/bt4222_project/Data/Models/Baseline


## 1 Load and check features and labels

Training data is loaded from featA_train_c0 through featA_train_c3 and the corresponding base_train_c0 through base_train_c3 files. Validation data is loaded from featA_val and base_val. No row-position alignment is assumed. A one-to-one outer join detects missing, additional or duplicate candidates. Only label-positive queries are evaluated, consistent with the project plan; excluded counts are printed.

Every candidate of a retained query stays in that query. The original list order and raw IDs never enter X. Keys determine a canonical starting order before a seeded shuffle, so changing the source file order does not change the experiment.


In [ ]:
def local_input(name):
    src = FEATURE_DIR / f'{name}.parquet'
    if not src.is_file():
        raise FileNotFoundError(f'{src} missing. Check the shared Drive link and finish FeatureEngineering first.')
    dst = LOCAL / src.name
    # Copy afresh: Drive files may have been rebuilt by a teammate.
    shutil.copy2(src, dst)
    return dst

def load_split(tags):
    tags = [tags] if isinstance(tags, str) else list(tags)
    tag = '+'.join(tags)
    f = pd.concat([pd.read_parquet(local_input(f'featA_{t}')) for t in tags], ignore_index=True)
    b = pd.concat([pd.read_parquet(local_input(f'base_{t}'), columns=KEYS + ['label']) for t in tags], ignore_index=True)
    for name, table in [('features', f), ('base', b)]:
        if table[KEYS].isna().any().any() or table.duplicated(KEYS).any():
            raise ValueError(f'{tag}: null or duplicate keys in {name}')
    absent = set(FEATURES) - set(f.columns)
    if absent:
        raise ValueError(f'{tag}: missing features {absent}')
    # Labels come exclusively from the base file; ignore any extra feature-file columns.
    d = b.merge(f[KEYS + FEATURES], on=KEYS, how='outer', validate='one_to_one', indicator=True)
    if not d['_merge'].eq('both').all():
        raise ValueError(f'{tag}: feature and label candidate sets differ; rebuild features.')
    d = d.drop(columns='_merge')
    if not d.label.isin([0, 1, 2, 3]).all():
        raise ValueError('Labels must be 0/1/2/3. Confirm the mapping in the dataset.')
    positive = d.groupby('user_id').label.transform('max').gt(0)
    excluded = d.loc[~positive, 'user_id'].nunique()
    print(tag, '| queries without positives excluded:', excluded)
    d = d.loc[positive].sort_values(KEYS).reset_index(drop=True)
    if d.empty:
        raise ValueError(f'{tag}: no usable queries')
    d['_shuffle'] = np.random.default_rng(SEED).random(len(d))
    d = d.sort_values(['user_id', '_shuffle']).drop(columns='_shuffle').reset_index(drop=True)
    d[FEATURES] = d[FEATURES].astype('float32')
    if np.isinf(d[FEATURES].to_numpy()).any():
        raise ValueError('Infinite feature values detected; NaN is allowed, infinity is not.')
    if d[['prod_pop_ord', 'prod_pop_cart', 'prod_pop_click', 'has_meta']].isna().any().any():
        raise ValueError('Popularity and has_meta must not be missing.')
    if (d[['prod_pop_ord', 'prod_pop_cart', 'prod_pop_click']] < 0).any().any():
        raise ValueError('Popularity log-counts must be nonnegative.')
    assert d.user_id.is_monotonic_increasing
    print(tag, '| rows:', len(d), '| queries:', d.user_id.nunique())
    return d

train = load_split([f'train_c{i}' for i in range(4)])     # all 120,174 usable training users
val = load_split('val')
assert set(train.user_id).isdisjoint(set(val.user_id)), 'Train/validation user overlap'
group_train = train.groupby('user_id', sort=False).size().to_numpy(dtype=np.int32)
group_val = val.groupby('user_id', sort=False).size().to_numpy(dtype=np.int32)
assert group_train.sum() == len(train) and group_val.sum() == len(val)
display(train[FEATURES].agg(['mean', 'min', 'max']).T.assign(missing_share=train[FEATURES].isna().mean()))


train_c0+train_c1+train_c2+train_c3 | queries without positives excluded: 0
train_c0+train_c1+train_c2+train_c3 | rows: 10750112 | queries: 120174
val | queries without positives excluded: 0
val | rows: 2294519 | queries: 25787


,mean,min,max,missing_share
query_coverage,0.721159,0.0,1.000000,0.004544
tfidf_cosine,0.263476,0.0,1.000000,0.004544
brand_name_match,0.049002,0.0,1.000000,0.057788
prod_pop_ord,0.380336,0.0,8.122372,0.000000
prod_pop_cart,0.661787,0.0,9.982299,0.000000
prod_pop_click,0.574545,0.0,9.060912,0.000000
has_meta,0.995456,0.0,1.000000,0.000000


## 2 Shared evaluation with reproducible tie breaking

NDCG@10 uses linear label gains [0,1,2,3]. MRR is the reciprocal position of the first label >=1 across the whole list. Hit@10 checks for at least one label >=1 in the first ten. Metrics are averaged equally over queries. This project excludes queries with no positives before evaluation.

The evaluator sorts equal scores with a seeded random priority assigned in canonical (user_id, wid) order, independently of labels. The same implementation also supplies the early-stopping metric. It will be saved for reuse by Model B.


In [ ]:
EVALUATION_CODE = 'import numpy as np\nimport pandas as pd\n\ndef make_evaluator(keys, labels, k=10, seed=42):\n    """Return a vectorized, tie-safe evaluator for contiguous query groups."""\n    keys = keys[[\'user_id\', \'wid\']].reset_index(drop=True)\n    y = np.asarray(labels, dtype=float)\n    if len(keys) != len(y) or keys.duplicated().any() or keys.isna().any().any():\n        raise ValueError(\'Invalid candidate keys\')\n    if not keys.user_id.is_monotonic_increasing:\n        raise ValueError(\'Queries must be contiguous and sorted by user_id\')\n    if not len(y) or not np.isin(y, [0, 1, 2, 3]).all():\n        raise ValueError(\'Invalid relevance labels\')\n    group, users = pd.factorize(keys.user_id, sort=False)\n    n = len(users)\n    counts = np.bincount(group, minlength=n)\n    starts = np.r_[0, np.cumsum(counts)[:-1]]\n    canonical = np.lexsort((keys.wid.to_numpy(), keys.user_id.to_numpy()))\n    tie = np.empty(len(y))\n    tie[canonical] = np.random.default_rng(seed).random(len(y))\n    def discounted(order):\n        ranks = np.arange(len(y)) - starts[group[order]]\n        keep = ranks < k\n        return np.bincount(group[order][keep],\n            weights=y[order][keep] / np.log2(ranks[keep] + 2), minlength=n)\n    ideal = np.lexsort((tie, -y, group))\n    idcg = discounted(ideal)\n    if not (idcg > 0).all():\n        raise ValueError(\'Exclude queries without positive candidates before evaluation\')\n    def evaluate(scores, per_query=False):\n        scores = np.asarray(scores, dtype=float)\n        if scores.shape != y.shape or not np.isfinite(scores).all():\n            raise ValueError(\'Scores must be finite and match the candidate rows\')\n        order = np.lexsort((tie, -scores, group))\n        ranked_group = group[order]\n        ranks = np.arange(len(y)) - starts[ranked_group]\n        positive = y[order] > 0\n        rr = np.zeros(n)\n        np.maximum.at(rr, ranked_group[positive], 1.0 / (ranks[positive] + 1))\n        hits = np.bincount(ranked_group[positive & (ranks < k)], minlength=n) > 0\n        table = pd.DataFrame({\'user_id\': users, \'ndcg10\': discounted(order) / idcg,\n                              \'mrr\': rr, \'hit10\': hits.astype(float)})\n        return table if per_query else table[[\'ndcg10\', \'mrr\', \'hit10\']].mean().to_dict()\n    return evaluate\n'
exec(EVALUATION_CODE)
evaluate_val = make_evaluator(val[KEYS], val.label.to_numpy(), k=K, seed=SEED)


## 3 Train Model A

Only the seven explicit features enter the model. Early stopping selects the iteration by the shared, tie-safe validation NDCG@10, after 50 rounds without improvement. There is no parameter search here. Model B should reuse these settings.

The subsample frequency is set to 1 so subsample=0.8 actually takes effect. Scores are relative ranking values, not click probabilities. Evaluation on millions of rows each round may take several minutes or longer; no GPU is required.


In [ ]:
PARAMS = dict(
    objective='lambdarank', metric='None', label_gain=[0, 1, 2, 3],
    n_estimators=1000, learning_rate=0.05, num_leaves=63,
    min_child_samples=100, colsample_bytree=0.8,
    subsample=0.8, subsample_freq=1, random_state=SEED,
    n_jobs=-1, verbosity=-1, deterministic=True, force_col_wise=True,
)

def validation_ndcg(y_true, y_pred):
    # Only this validation dataset is passed to fit.
    if not np.array_equal(y_true, val.label.to_numpy()):
        raise ValueError('Evaluation labels do not match the validation rows')
    return 'ndcg10_tie_safe', evaluate_val(y_pred)['ndcg10'], True

model = lgb.LGBMRanker(**PARAMS)
model.fit(
    train[FEATURES], train.label.to_numpy(), group=group_train,
    eval_set=[(val[FEATURES], val.label.to_numpy())],
    eval_group=[group_val], eval_names=['validation'],
    eval_metric=validation_ndcg,
    callbacks=[lgb.early_stopping(50, first_metric_only=True), lgb.log_evaluation(25)],
)
print('Best iteration:', model.best_iteration_)
scores_model = model.predict(val[FEATURES], num_iteration=model.best_iteration_)


Training until validation scores don't improve for 50 rounds
[25]	validation's ndcg10_tie_safe: 0.346783
[50]	validation's ndcg10_tie_safe: 0.347725
[75]	validation's ndcg10_tie_safe: 0.34858
[100]	validation's ndcg10_tie_safe: 0.349134
[125]	validation's ndcg10_tie_safe: 0.34932
[150]	validation's ndcg10_tie_safe: 0.349456
[175]	validation's ndcg10_tie_safe: 0.34939
[200]	validation's ndcg10_tie_safe: 0.349629
[225]	validation's ndcg10_tie_safe: 0.349768
Early stopping, best iteration is:
[186]	validation's ndcg10_tie_safe: 0.350167
Evaluated only: ndcg10_tie_safe
Best iteration: 186


## 4 Compare random, popularity and the learned baseline

Recover counts using expm1 before adding the three popularity signals; adding their log values would define a different reference ranking. Minor float32 rounding in recovered counts is removed with rint.

All references use identical validation candidates and the same tie-safe evaluator. Random is one reproducible draw, so its exact number can vary with another seed. A trained ranker need not beat popularity; an unexpectedly weak result warrants investigation, not an automatic conclusion that there is a bug.


In [ ]:
rng = np.random.default_rng(SEED + 1)
scores_random = rng.random(len(val))
count_columns = ['prod_pop_ord', 'prod_pop_cart', 'prod_pop_click']
scores_popularity = np.rint(np.expm1(val[count_columns].to_numpy(dtype=np.float64))).sum(axis=1)
results = pd.DataFrame({
    'Random': evaluate_val(scores_random),
    'Popularity only': evaluate_val(scores_popularity),
    'Model A': evaluate_val(scores_model),
}).T
results.index.name = 'ranking'
display(results)
print('Model A minus popularity, NDCG@10:',
      results.loc['Model A', 'ndcg10'] - results.loc['Popularity only', 'ndcg10'])
importance = pd.DataFrame({
    'feature': FEATURES,
    'gain': model.booster_.feature_importance(importance_type='gain'),
    'splits': model.booster_.feature_importance(importance_type='split'),
}).sort_values('gain', ascending=False)
display(importance)


,ndcg10,mrr,hit10
ranking,,,
Random,0.085357,0.101618,0.226742
Popularity only,0.339190,0.333674,0.638112
Model A,0.350167,0.345115,0.649048


Model A minus popularity, NDCG@10: 0.010977361589928514


,feature,gain,splits
4,prod_pop_cart,1.512500e+06,1708
5,prod_pop_click,5.356657e+05,2197
1,tfidf_cosine,1.592339e+05,3484
3,prod_pop_ord,6.976352e+04,1728
0,query_coverage,5.526911e+04,1691
2,brand_name_match,1.461062e+04,724
6,has_meta,0.000000e+00,0


## 5 Save Model A

Existing baseline output files are replaced when this cell is run. The saved JSON fixes features, parameters, metric definitions and data fingerprints for the comparison with Model B. The keyed candidate and per-query prediction files support later paired comparisons. No test metrics are produced.

For Model B, keep the exact same candidate keys, labels, splits, baseline features, gains, evaluator and parameters. Add history features and compare paired per-query NDCG@10. Final test evaluation is performed in ModelComparison.ipynb.

In [ ]:
def fingerprint(frame):
    ordered = frame.sort_values(KEYS).reset_index(drop=True)
    hashes = pd.util.hash_pandas_object(ordered, index=False).to_numpy()
    return hashlib.sha256(hashes.tobytes()).hexdigest()

SAVE_DIR.mkdir(parents=True, exist_ok=True)
model.booster_.save_model(str(LOCAL / 'model_A.txt'), num_iteration=model.best_iteration_)
results.to_csv(LOCAL / 'validation_metrics.csv')
importance.to_csv(LOCAL / 'feature_importance.csv', index=False)
predictions = val[KEYS + ['label']].copy()
predictions['score_A'] = scores_model
predictions['score_popularity'] = scores_popularity
predictions.to_parquet(LOCAL / 'validation_predictions.parquet', index=False)
evaluate_val(scores_model, per_query=True).to_parquet(LOCAL / 'validation_per_query.parquet', index=False)
(LOCAL / 'ranking_evaluation.py').write_text(EVALUATION_CODE)
metadata = dict(
    stage='development_validation', features=FEATURES, parameters=PARAMS,
    seed=SEED, best_iteration=int(model.best_iteration_),
    rows={'train': len(train), 'val': len(val)},
    queries={'train': len(group_train), 'val': len(group_val)},
    metrics=results.to_dict(orient='index'),
    metric_definitions={'ndcg10': 'linear gains [0,1,2,3], top 10',
                        'mrr': 'first label >=1, full list', 'hit10': 'any label >=1, top 10'},
    no_positive_queries='excluded', tie_break='canonical keys then seeded random priority',
    versions={'python': platform.python_version(), 'numpy': np.__version__,
              'pandas': pd.__version__, 'lightgbm': lgb.__version__},
    fingerprints={
        'train_keys_labels': fingerprint(train[KEYS + ['label']]),
        'val_keys_labels': fingerprint(val[KEYS + ['label']]),
        'train_features': fingerprint(train[KEYS + FEATURES]),
        'val_features': fingerprint(val[KEYS + FEATURES]),
    },
)
(LOCAL / 'run_config.json').write_text(json.dumps(metadata, indent=2))
for name in ['model_A.txt', 'validation_metrics.csv', 'feature_importance.csv',
             'validation_predictions.parquet', 'validation_per_query.parquet',
             'ranking_evaluation.py', 'run_config.json']:
    shutil.copy2(LOCAL / name, SAVE_DIR / name)
print('Saved development result to:', SAVE_DIR)


Saved development result to: /content/drive/MyDrive/bt4222_project/Data/Models/Baseline


LightGBM API references: [LGBMRanker](https://lightgbm.readthedocs.io/en/stable/pythonapi/lightgbm.LGBMRanker.html) and [early stopping](https://lightgbm.readthedocs.io/en/stable/pythonapi/lightgbm.early_stopping.html).
